# Tagesschau Umfrage processing

## Umfrage AfD

Which questions (chart titles) and answer items (categories) from `tagesschau_umfrage-afd.csv` were asked in which election year? Only the wording is compared, not the values, to judge comparability across 2013, 2017, 2021 and 2025.

In [ ]:
# Compare which questions (chart titles) and answer items (categories) appear in which election year.
# Only the text is compared, values are ignored. Goal: find what is comparable across years.
import re
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd
from IPython.display import display

afd = pd.read_csv(Path("../data/interim/tagesschau_umfrage-afd.csv"))
afd["year"] = afd["election"].str[:4]  # "2025-02-23-BT-DE" -> "2025"
years = sorted(afd["year"].unique())


def norm(text):
    """Normalise wording so that only real differences remain."""
    t = str(text).lower()
    t = re.sub(r"[„“”\"'…]", "", t)          # quotation marks and ellipsis
    t = t.replace("wählende", "wähler")      # 2021+ uses gender-neutral "Wählende", before "Wähler"
    t = re.sub(r"[.:,]+\s*$", "", t.strip())  # trailing punctuation ("Kriminalitätsbekämpfung:  ")
    return re.sub(r"\s+", " ", t).strip()


afd["chart_norm"] = afd["chart"].map(norm)
afd["item_norm"] = afd["category"].map(norm)

# 1) Questions (chart titles): in which years does each one appear? (identical after normalising)
charts = afd.groupby(["chart_norm", "year"]).size().unstack(fill_value=0).gt(0)
charts["n_years"] = charts.sum(axis=1)
print("1) Chart titles asked in more than one year (exact after normalising):")
display(charts[charts["n_years"] > 1].sort_values("n_years", ascending=False).replace({True: "x", False: ""}))


# Helper: pairs of texts from *different* years whose wording is similar but not identical
def near_matches(df, col, min_ratio=0.75):
    unique = df[[col, "year"]].drop_duplicates().values.tolist()
    rows = []
    for i, (a, ya) in enumerate(unique):
        for b, yb in unique[i + 1:]:
            if ya == yb or a == b:
                continue
            ratio = SequenceMatcher(None, a, b).ratio()
            if ratio >= min_ratio:
                rows.append({"year_a": ya, "text_a": a, "year_b": yb, "text_b": b, "similarity": round(ratio, 2)})
    return pd.DataFrame(rows).sort_values("similarity", ascending=False) if rows else pd.DataFrame()


print("\n2) Chart titles with similar but not identical wording across years (check by hand):")
display(near_matches(afd, "chart_norm"))

# 3) Answer items / statements: which ones are repeated across years, and under which chart?
#    Short demographic labels ("alle", "männer", "arbeiter") also match; the chart column shows the context.
items = (afd.groupby("item_norm")
            .agg(years=("year", lambda s: ", ".join(sorted(s.unique()))),
                 n_years=("year", "nunique"),
                 charts=("chart", lambda s: " || ".join(s.unique())))
            .query("n_years > 1")
            .sort_values("n_years", ascending=False))
print("\n3) Answer items asked in more than one year (exact after normalising):")
display(items)

print("\n4) Answer items with similar but not identical wording across years (check by hand):")
display(near_matches(afd, "item_norm", min_ratio=0.8))

print("\nNote: the 2013 'umfrage-afd' URL serves the general 'Aktuelle Themen' page;"
      " only 'Ansichten über die AfD' is AfD-specific there.")

In [2]:
# List the statements asked under "Ansichten über die AfD" per election year (no text comparison).
import pandas as pd

afd = pd.read_csv("../data/interim/tagesschau_umfrage-afd.csv")
views = afd[afd["chart"] == "Ansichten über die AfD"]

for election, group in views.groupby("election", sort=True):
    # 2025 repeats three statements in the alt text (8 rows, 5 distinct), so list each statement once
    statements = group["category"].drop_duplicates()
    print(f"{election[:4]} ({len(statements)} statements)")
    for s in statements:
        print("  -", s)
    print()

# Note: 2021 has no "Ansichten über die AfD" chart on this page

2013 (4 statements)
  - „Löst zwar keine Probleme, nennt die Dinge aber beim Namen“
  - „Ist eine Alternative für die, die sonst gar nicht wählen würden“
  - „Ich finde gut, dass eine Partei sich gegen den Euro ausspricht“
  - „Ist keine ernstzunehmende Partei“

2017 (4 statements)
  - „Hat besser als andere verstanden, dass sich viele Menschen nicht mehr sicher fühlen.“
  - „Finde es gut, dass sie den Einfluss des Islam in Deutschland verringern will.“
  - „Finde es gut, dass sie den Zuzug von Flüchtlingen stärker begrenzen will.“
  - „Distanziert sich nicht genug von rechtsextremen Positionen.“

2025 (5 statements)
  - „Distanziert sich nicht genug von rechtsextremen Positionen.“
  - „Habe Angst, dass sie unsere Demokratie zerstört.“
  - „Hat besser als andere Parteien verstanden, dass sich viele Menschen nicht mehr sicher fühlen.“
  - „Finde es gut, dass sie den Zuzug von Ausländern und Flüchtlingen stärker begrenzen will.“
  - „Alice Weidel spricht in vielen Punkten klar aus, was S

# Decision on tagesschau-umfrage-afd.csv

Hat keine richtige Vergleichbarkeit, denn es wurden über die Jahre andere Fragen gestellt, wir lassen also diesen Datensatz weg